# CODEX and H&E Image Registration & Merging Pipeline

This notebook demonstrates the complete workflow for registering and merging CODEX multiplex immunofluorescence images with H&E histology images. The pipeline produces high-quality pyramidal OME-TIFF files suitable for downstream analysis and visualization.

## Key Features:
- **Memory-efficient processing**: Handles large multiplex images with optimized memory usage
- **Affine registration**: Uses palom library for robust H&E to CODEX DAPI registration
- **Pyramidal OME-TIFF output**: Creates multi-resolution images compatible with QuPath and other viewers
- **Metadata preservation**: Maintains channel names, colors, and spatial calibration
- **Modular design**: Organized codebase with reusable utility functions

## Pipeline Overview:
1. **Setup and Configuration** - Define paths and parameters
2. **Input Validation** - Verify available image files
3. **H&E Preprocessing** - Resize to match CODEX resolution
4. **DAPI Extraction** - Extract reference channel from CODEX
5. **Image Registration** - Align H&E to CODEX coordinate system
6. **Channel Merging** - Combine all channels into single file
7. **Output Generation** - Save as pyramidal OME-TIFF

## 1. Project Directory Structure Overview

The SPACEMAP project follows a well-organized structure that separates data, code, and outputs:

```
SPACEMAP/
├── data/                               # Input data directory
│   ├── codex/                          # CODEX multiplex images (.qptiff)
│   ├── he/                             # H&E histology images (.qptiff)
│   └── boundaries/                     # ROI boundary files (.geojson)
│
├── notebooks/                          # Jupyter notebooks
│   ├── Step1_registration.ipynb        # Registration and merging (this notebook)
│   ├── Step2_roi_extraction.ipynb      # ROI extraction workflow
│   ├── Step3A_segmentation_cellpose    # Cell segmentation with Cellpose
│   ├── Step3B_segmentation_deepcell    # Cell segmentation with DeepCell
│   ├── Step4_annotation                # Cell type annotation
│   ├── Step5_batch_correction          # Batch effect correction
│   ├── Step6A_phenotype_leiden         # Phenotyping with Leiden clustering
│   ├── Step6B_phenotype_SOM            # Phenotyping with SOM
│   ├── Step6C_phenotype_threshold      # Threshold-based phenotyping
│   └── Step6D_phenotype_SCIMAP         # SCIMAP-based phenotyping
│
├── src/                                # Source code modules
│   ├── __init__.py                     # Package initialization
│   ├── utils.py                        # Utility functions (logging, helpers)
│   └── registration.py                 # Registration pipeline functions
│
└── outputs/                            # Generated outputs
    └── registration/                   # Registration pipeline outputs
        ├── preprocessed/               # Intermediate processing files
        │   ├── he_resized/             # Resized H&E images for registration
        │   └── codex_dapi/             # Extracted CODEX DAPI channels
        ├── registered_he/              # H&E images registered to CODEX space
        └── merged/                     # Final merged OME-TIFF files
```

### Key Directories:
- **`data/`**: Place your input CODEX and H&E images here (naming: `*_he.qptiff`, `*_codex.qptiff`)
- **`src/`**: Contains modular, reusable functions imported by notebooks
- **`outputs/registration/`**: All registration pipeline outputs organized by processing stage
- **`notebooks/`**: Interactive analysis workflows with modular design

## 2. Import Required Libraries and Utility Functions

Import all necessary libraries and our custom utility functions from the `src/` directory. 

The modular design separates core functionality into reusable modules:
- **`utils.py`**: Common utilities like logging and helper functions
- **`registration.py`**: Core registration pipeline functions moved from notebook for reusability

In [ ]:
import os
import sys
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt
project_root = Path('..').resolve()
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))
from spacemap.utils import log_time
from spacemap.background_subtraction_utils import (
    get_channel_records,
    get_channel_info,
    get_image_mpp,
    normalize_image_stem,
    is_supported_image_file,
    read_channel_image,
    )
from spacemap.registration import (
    preprocess_he_image,
    extract_dapi_channel,
    register_images,
    merge_channels,
    resolve_reference_channel,
    )

print("All libraries imported successfully!")
print(f"Project root: {project_root}")
print(f"Source path: {src_path}")

## 3. Set Directory Paths and Parameters

Configure all input/output directories and key parameters for the registration pipeline.

The pipeline uses a structured output directory approach:
- **`outputs/registration/`**: Main output directory for this workflow
- **`preprocessed/`**: Intermediate files (resized H&E, extracted DAPI)
- **`registered_he/`**: H&E images transformed to CODEX coordinate space
- **`merged/`**: Final pyramidal OME-TIFF files with all channels

In [ ]:
# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================
log_time("Setting up directories and parameters...")

# Input directories (relative to notebook location)
codex_imgs_dir = '../data/codex/'
he_imgs_dir = '../data/he'

# Output directories
outputs_dir = '../outputs/registration'
preprocessed_dir = os.path.join(outputs_dir, 'preprocessed')
he_resized_dir = os.path.join(preprocessed_dir, 'he_resized')
codex_dapi_dir = os.path.join(preprocessed_dir, 'codex_dapi')
registered_dir = os.path.join(outputs_dir, 'registered_he')
ometiff_dir = os.path.join(outputs_dir, 'merged')

# Create output directories if they don't exist
for directory in [preprocessed_dir, he_resized_dir, codex_dapi_dir, registered_dir, ometiff_dir]:
    os.makedirs(directory, exist_ok=True)
    print(f" Created/verified directory: {directory}")

def detect_mpp_if_available(path, fallback):
    if path and os.path.exists(path):
        return float(get_image_mpp(path, default=fallback).get('codex', fallback))
    return float(fallback)

# =============================================================================
# PHYSICAL PIXEL SIZES (MICRONS PER PIXEL)
# =============================================================================
# Defaults are used when metadata is missing. For OME-TIFF inputs these values are
# auto-detected per file during processing.
mpp = {
    'hne': 0.2495,
    'codex': 0.5083
}

# =============================================================================
# FILE NAMING CONVENTIONS
# =============================================================================
CODEX_SUFFIX = '_codex'  # e.g., P11_codex.qptiff or P11_codex.ome.tiff
HE_SUFFIX = '_he'        # e.g., P11_he.qptiff or P11_he.ome.tiff

# =============================================================================
# REGISTRATION ANCHOR CONFIGURATION
# =============================================================================
# Leave DEFAULT_REFERENCE_CHANNEL = None to use automatic detection.
# Set a channel name such as 'Hoechst' to use it for all pairs by default.
DEFAULT_REFERENCE_CHANNEL = None

# Optional per-sample overrides keyed by normalized sample prefix.
# Example: {'P37_S29_A24_C59kX_E15_20220106_014304_946511-zlib': 'Hoechst'}
REFERENCE_CHANNEL_BY_PREFIX = {}

def get_reference_override(codex_path):
    sample_name = normalize_image_stem(codex_path)
    if CODEX_SUFFIX and sample_name.endswith(CODEX_SUFFIX):
        sample_name = sample_name[:-len(CODEX_SUFFIX)]
    return REFERENCE_CHANNEL_BY_PREFIX.get(sample_name, DEFAULT_REFERENCE_CHANNEL)

# =============================================================================
# CHANNEL FILTERING
# =============================================================================
# List of channels to keep in the final output. Set to None or empty list to keep all.
# Example: CHANNELS_TO_KEEP = ['DAPI', 'CD8', 'CD4', 'CK']
CHANNELS_TO_KEEP = [] 

# =============================================================================
# PROCESSING PARAMETERS
# =============================================================================
pyramid_params = {
    'tile_size': (256, 256),
    'n_pyramid_levels': 2,
    'compression': 'deflate'
}

print(f"\n Resolution Defaults:")
print(f"   H&E default pixel size: {mpp['hne']} μm/pixel")
print(f"   CODEX default pixel size: {mpp['codex']} μm/pixel")
print(f"   Default anchor marker: {DEFAULT_REFERENCE_CHANNEL or 'Auto'}")

print(f"\n Directory structure verified successfully!")
print(f"   Input directories: {codex_imgs_dir}, {he_imgs_dir}")
print(f"   Output directory: {outputs_dir}")

## 4. List Available Input Images

Scan the input directories to identify available CODEX and H&E image pairs for processing.

In [ ]:
# =============================================================================
# DISCOVER AVAILABLE IMAGE PAIRS
# =============================================================================
log_time("Scanning for available image files...")

def find_files_with_suffix(directory, suffix):
    """Find files ending with suffix (ignoring extension)."""
    matches = []
    if not os.path.exists(directory):
        return matches
        
    for f in os.listdir(directory):
        if not is_supported_image_file(f):
            continue
        
        name_no_ext = normalize_image_stem(f)
        if name_no_ext.endswith(suffix):
            prefix = name_no_ext[:-len(suffix)]
            matches.append((prefix, f))
    return matches

# Find H&E files
he_matches = find_files_with_suffix(he_imgs_dir, HE_SUFFIX)
he_dict = {prefix: f for prefix, f in he_matches}

# Find CODEX files
codex_matches = find_files_with_suffix(codex_imgs_dir, CODEX_SUFFIX)
codex_dict = {prefix: f for prefix, f in codex_matches}

# Match pairs
image_pairs = []
for prefix, he_file in he_dict.items():
    if prefix in codex_dict:
        codex_file = codex_dict[prefix]
        image_pairs.append((he_file, codex_file))
    else:
        print(f"Warning: Missing CODEX pair for {he_file} (Prefix: {prefix})")

# Check for orphan CODEX files
for prefix, codex_file in codex_dict.items():
    if prefix not in he_dict:
        print(f"Warning: Missing H&E pair for {codex_file} (Prefix: {prefix})")

# Display results
print(f"\nImage Discovery Results:")
print(f"   Found {len(he_matches)} H&E files")
print(f"   Found {len(codex_matches)} CODEX files")
print(f"   Found {len(image_pairs)} complete image pairs")

if image_pairs:
    print(f"\nAvailable Image Pairs:")
    for i, (he_file, codex_file) in enumerate(image_pairs, 1):
        prefix = normalize_image_stem(he_file)[:-len(HE_SUFFIX)]
        print(f"   {i}. {prefix}")
        print(f"      H&E:   {he_file}")
        print(f"      CODEX: {codex_file}")
        
    print(f"\nWill process all {len(image_pairs)} pairs")
    
else:
    print("No complete image pairs found!")
    print("   Please ensure your files follow the naming convention:")
    print(f"   - H&E files: *{HE_SUFFIX}.qptiff/ome.tif/ome.tiff")
    print(f"   - CODEX files: *{CODEX_SUFFIX}.qptiff/ome.tif/ome.tiff")

## 5. H&E Image Preprocessing (Batch)

Resize all H&E images to match CODEX resolution for proper registration.

In [ ]:
# =============================================================================
# H&E IMAGE PREPROCESSING (BATCH)
# =============================================================================

# Configuration
BATCH_PROCESSING = True  # Set to False to process only the first pair for testing

# Build full paths from the image_pairs (which contains only filenames)
full_image_pairs = []
for he_file, codex_file in image_pairs:
    he_path = os.path.join(he_imgs_dir, he_file)
    codex_path = os.path.join(codex_imgs_dir, codex_file)
    full_image_pairs.append((he_path, codex_path))

print(f"Found {len(full_image_pairs)} image pairs to process")

# Determine which pairs to process
if BATCH_PROCESSING:
    pairs_to_process = full_image_pairs
    print("BATCH MODE: Processing all available pairs")
else:
    pairs_to_process = full_image_pairs[:1]  # Process only first pair
    print("TEST MODE: Processing only the first pair")

if image_pairs:
    log_time("Starting H&E preprocessing for all image pairs...")
    
    he_processed_files = []
    for idx, (he_path, codex_path) in enumerate(pairs_to_process, 1):
        base_name = normalize_image_stem(he_path).replace('_he', '')
        pair_mpp = {
            'hne': detect_mpp_if_available(he_path, mpp['hne']),
            'codex': detect_mpp_if_available(codex_path, mpp['codex']),
        }
        print(f"\n[{idx}/{len(pairs_to_process)}] Processing H&E: {base_name}")
        print(f"   Detected MPP -> H&E: {pair_mpp['hne']:.4f}, CODEX: {pair_mpp['codex']:.4f}")
        
        try:
            he_resized_path = preprocess_he_image(he_path, he_resized_dir, pair_mpp)
            he_processed_files.append((he_resized_path, codex_path, base_name, pair_mpp))
            print(f"   H&E preprocessed: {Path(he_resized_path).name}")
            
        except Exception as e:
            print(f"   Error processing H&E for {base_name}: {str(e)}")
            continue
    
    print(f"\n{'='*60}")
    print(f"H&E preprocessing completed!")
    print(f"Processed {len(he_processed_files)} H&E images successfully")
    
    if he_processed_files:
        print(f"\nSample H&E preprocessing result:")
        sample_he_path = he_processed_files[0][0]
        sample_base_name = he_processed_files[0][2]
        
        try:
            he_resized = tifffile.imread(sample_he_path)
            plt.figure(figsize=(12, 8))
            if len(he_resized.shape) == 3:
                plt.imshow(he_resized)
            else:
                plt.imshow(he_resized, cmap='gray')
            plt.title(f"Sample Resized H&E Image: {sample_base_name}\\nShape: {he_resized.shape}")
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Could not display sample image: {e}")
else:
    print("No image pairs available for processing")
    he_processed_files = []

## 6. CODEX DAPI Channel Extraction (Batch)

Extract DAPI channels from all CODEX images to serve as registration references.

In [ ]:
# =============================================================================
# CODEX REFERENCE CHANNEL EXTRACTION (BATCH)
# =============================================================================

if he_processed_files:
    log_time("Starting CODEX reference extraction for all image pairs...")
    
    dapi_extracted_files = []
    for idx, (he_resized_path, codex_path, base_name, pair_mpp) in enumerate(he_processed_files, 1):
        reference_override = get_reference_override(codex_path)
        reference_label = reference_override or 'Auto'
        print(f"\n[{idx}/{len(he_processed_files)}] Extracting reference: {base_name}")
        print(f"   Anchor marker: {reference_label}")
        
        try:
            dapi_path = extract_dapi_channel(
                codex_path,
                codex_dapi_dir,
                reference_channel=reference_override,
            )
            dapi_extracted_files.append((
                he_resized_path,
                dapi_path,
                codex_path,
                base_name,
                pair_mpp,
                reference_override,
            ))
            print(f"   Reference extracted: {Path(dapi_path).name}")
            
        except Exception as e:
            print(f"   Error extracting reference for {base_name}: {str(e)}")
            continue
    
    print(f"\n{'='*60}")
    print(f"Reference extraction completed!")
    print(f"Processed {len(dapi_extracted_files)} reference channels successfully")
    
    if dapi_extracted_files:
        print(f"\nSample reference extraction result:")
        sample_he_path, sample_dapi_path, sample_codex_path, sample_base_name, sample_pair_mpp, sample_reference_override = dapi_extracted_files[0]
        
        try:
            if os.path.abspath(sample_dapi_path) == os.path.abspath(sample_codex_path):
                channel_names, _, _, _ = get_channel_info(sample_dapi_path)
                sample_ref_index, sample_ref_name = resolve_reference_channel(
                    channel_names,
                    reference_channel=sample_reference_override,
                )
                dapi_img = read_channel_image(sample_dapi_path, sample_ref_index)
                ref_label = sample_ref_name or 'Channel 0'
            else:
                dapi_img = tifffile.imread(sample_dapi_path)
                ref_label = sample_reference_override or 'Auto'
            
            plt.figure(figsize=(12, 8))
            plt.imshow(dapi_img, cmap='gray')
            plt.title(f"Sample CODEX Reference: {sample_base_name}\\nAnchor: {ref_label} | Shape: {dapi_img.shape} | Range: {dapi_img.min()}-{dapi_img.max()}")
            plt.colorbar(label='Intensity')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Could not display sample reference image: {e}")
else:
    print("No H&E processed files available for reference extraction")
    dapi_extracted_files = []

## 7. H&E to CODEX Registration (Batch)

Register all H&E images to their corresponding CODEX DAPI channels using affine transformations.

In [ ]:
# =============================================================================
# H&E TO CODEX REGISTRATION (BATCH)
# =============================================================================

if dapi_extracted_files:
    log_time("Starting H&E to CODEX registration for all image pairs...")
    
    registered_files = []
    for idx, (he_resized_path, dapi_path, codex_path, base_name, pair_mpp, reference_override) in enumerate(dapi_extracted_files, 1):
        reference_label = reference_override or 'Auto'
        print(f"\n[{idx}/{len(dapi_extracted_files)}] Registering: {base_name}")
        print(f"   Anchor marker: {reference_label}")
        
        try:
            reg_mpp = {'codex': pair_mpp['codex'], 'hne': pair_mpp['codex']}
            registered_he_path = register_images(
                he_resized_path,
                dapi_path,
                registered_dir,
                reg_mpp,
                reference_channel=reference_override,
            )
            registered_files.append((registered_he_path, codex_path, base_name, pair_mpp, reference_override))
            print(f"   Registration complete: {Path(registered_he_path).name}")
            
        except Exception as e:
            print(f"   Error registering {base_name}: {str(e)}")
            continue
    
    print(f"\n{'='*60}")
    print(f"Registration completed!")
    print(f"Processed {len(registered_files)} registrations successfully")
    
    if registered_files:
        print(f"\nSample registration result:")
        sample_reg_path = registered_files[0][0]
        sample_base_name = registered_files[0][2]
        sample_reference_override = registered_files[0][4]
        
        try:
            sample_dapi_path = None
            sample_codex_path = None
            for he_path, dapi_path, codex_path, base, pair_mpp, reference_override in dapi_extracted_files:
                if base == sample_base_name:
                    sample_dapi_path = dapi_path
                    sample_codex_path = codex_path
                    sample_reference_override = reference_override
                    break
            
            if sample_dapi_path and os.path.exists(sample_dapi_path):
                with tifffile.TiffFile(sample_reg_path) as tif:
                    first_channel = tif.pages[0].asarray()
                
                if os.path.abspath(sample_dapi_path) == os.path.abspath(sample_codex_path):
                    channel_names, _, _, _ = get_channel_info(sample_dapi_path)
                    sample_ref_index, sample_ref_name = resolve_reference_channel(
                        channel_names,
                        reference_channel=sample_reference_override,
                    )
                    dapi_ref = read_channel_image(sample_dapi_path, sample_ref_index)
                    ref_label = sample_ref_name or 'Channel 0'
                else:
                    dapi_ref = tifffile.imread(sample_dapi_path)
                    ref_label = sample_reference_override or 'Auto'
                
                plt.figure(figsize=(15, 5))
                
                plt.subplot(1, 2, 1)
                plt.imshow(dapi_ref, cmap='gray')
                plt.title(f'CODEX Reference: {sample_base_name}\\nAnchor: {ref_label}')
                plt.axis('off')
                
                plt.subplot(1, 2, 2)
                plt.imshow(first_channel, cmap='gray')
                plt.title(f'H&E Registered: {sample_base_name}')
                plt.axis('off')
                
                plt.tight_layout()
                plt.show()
            else:
                print(f"Could not find reference image for comparison")
                
        except Exception as e:
            print(f"Could not display sample registration: {e}")
else:
    print("No reference files available for registration")
    registered_files = []

## 8. Channel Merging and Final Output (Batch)

Merge all CODEX channels with registered H&E channels to create final pyramidal OME-TIFF files.

In [ ]:
# =============================================================================
# CHANNEL MERGING AND FINAL OUTPUT (BATCH)
# =============================================================================

if registered_files:
    log_time("Starting channel merging for all image pairs...")
    
    final_outputs = []
    for idx, (registered_he_path, codex_path, base_name, pair_mpp, reference_override) in enumerate(registered_files, 1):
        print(f"\n[{idx}/{len(registered_files)}] Merging channels: {base_name}")
        
        try:
            final_output = merge_channels(registered_he_path, codex_path, ometiff_dir, base_name, pair_mpp, channels_to_keep=CHANNELS_TO_KEEP)
            final_outputs.append((final_output, base_name))
            print(f"   Merging complete: {Path(final_output).name}")
            
            if os.path.exists(final_output):
                file_size_gb = os.path.getsize(final_output) / (1024**3)
                print(f"      File size: {file_size_gb:.2f} GB")
            
        except Exception as e:
            print(f"   Error merging {base_name}: {str(e)}")
            continue
    
    print(f"\n{'='*60}")
    print(f"Channel merging completed!")
    print(f"Generated {len(final_outputs)} final merged images successfully")
    
    if final_outputs:
        print(f"\nFinal Output Summary:")
        total_size = 0
        for final_output, base_name in final_outputs:
            if os.path.exists(final_output):
                file_size = os.path.getsize(final_output) / (1024**3)
                total_size += file_size
                print(f"  {base_name}: {file_size:.2f} GB")
        
        print(f"\nTotal output size: {total_size:.2f} GB")
        print(f"Average file size: {total_size/len(final_outputs):.2f} GB")
        print(f"Output directory: {ometiff_dir}")
        
        print(f"\nSample merged image analysis:")
        sample_output = final_outputs[0][0]
        sample_base_name = final_outputs[0][1]
        
        try:
            with tifffile.TiffFile(sample_output) as tif:
                n_pages = len(tif.pages)
                print(f"Sample file: {sample_base_name}")
                print(f"  Number of channels: {n_pages}")

            channel_records, _ = get_channel_records(sample_output)
            if channel_records:
                print(f"  Channel Information:")
                for i, record in enumerate(channel_records[:10], 1):
                    meta = record['metadata']
                    color_hex = '#{:02X}{:02X}{:02X}'.format(*record['color'])
                    print(
                        f"    {i}. {record['name']} "
                        f"(Color: {color_hex}, Dye: {meta.get('dye', '-')}, "
                        f"Exp: {meta.get('exposure', '-')}, Em: {meta.get('emission_wavelength', '-')})"
                    )

                if len(channel_records) > 10:
                    print(f"    ... and {len(channel_records) - 10} more channels")
                        
        except Exception as e:
            print(f"Could not analyze sample merged image: {e}")
            
else:
    print("No registered files available for merging")
    final_outputs = []

## 9. Pipeline Summary and Next Steps

### Completed Tasks:
1. **Directory Structure Setup** - Organized project with modular architecture
2. **Function Organization** - Moved processing functions to `src/registration.py` module
3. **Input Validation** - Identified and validated image pairs
4. **Batch Processing Pipeline**:
   - **H&E Preprocessing** - Resized H&E to match CODEX resolution
   - **DAPI Extraction** - Extracted reference channel from CODEX with enhanced format handling
   - **Image Registration** - Aligned H&E to CODEX coordinate system using palom
   - **Channel Merging** - Combined all channels into single pyramidal OME-TIFF
   - **Quality Control** - Sample visualizations at each processing step

### Generated Files (for each sample pair):
- **Preprocessed H&E**: `outputs/registration/preprocessed/he_resized/{sample}_he_resized.tiff`
- **CODEX DAPI**: `outputs/registration/preprocessed/codex_dapi/{sample}_dapi.tiff`
- **Registered H&E**: `outputs/registration/registered_he/{sample}_he_registered.ome.tiff`
- **Final Merged**: `outputs/registration/merged/{sample}_merged.ome.tiff`

### Key Features of This Modular Design:
- **Automatic pair detection** - Processes all available image pairs automatically
- **Resume capability** - Skips already processed files, enabling interrupted workflow resumption
- **Error handling** - Continues processing remaining pairs if individual pairs fail
- **Progress tracking** - Shows detailed progress and statistics for each processing step
- **Modular functions** - Each step uses imported functions from `src/registration.py`
- **Enhanced DAPI handling** - Robust detection of different CODEX image formats
- **Sample visualization** - Representative results displayed at each major step

### Processing Statistics:
- **Total image pairs processed**: Check the output above for exact counts
- **Success rate**: Compare input pairs vs final outputs in each step
- **File sizes**: Total and average sizes displayed above for final outputs
- **Processing time**: Logged timestamps for each major processing step

### Next Steps:
1. **Quality Control**: Visually inspect registration quality for all pairs in QuPath or similar viewer
2. **ROI Extraction**: Use the companion notebook `Step2_roi_extraction.ipynb` to extract regions of interest from all merged images
3. **Segmentation**: Proceed to Step3 notebooks for cell segmentation
4. **Analysis**: Continue with downstream spatial analysis using the merged images

### Tips for Success:
- **File Naming**: Ensure consistent naming convention (`*_he.qptiff` and `*_codex.qptiff`)
- **Memory Management**: The batch processing handles memory efficiently with modular functions
- **Resume Processing**: Re-run any individual section to resume interrupted processing
- **Quality Check**: Always visually verify registration quality before proceeding
- **Backup**: Keep original files safe as processing creates new organized outputs
- **Function Reuse**: The `src/registration.py` module can be imported by other notebooks

### Troubleshooting:
- **Registration Fails**: Check that DAPI channel is clearly visible and well-focused
- **Memory Issues**: Close other applications to free up RAM during processing
- **Channel Mismatch**: Verify channel names in CODEX metadata
- **File Not Found**: Check directory paths and file naming conventions
- **Partial Processing**: Each step shows success/failure statistics
- **Dimension Issues**: The pipeline now handles different CODEX image formats automatically
- **Import Errors**: Ensure `src/` directory is properly structured with required modules